# Kaggle Training Notebook

This notebook uses the project Hydra config, prepares AudioMNIST on Kaggle, runs training, evaluates on test, and saves artifacts to `/kaggle/working/artifacts`.

Expected setup:
- the repository code is available on Kaggle
- either attach a pre-saved Hugging Face dataset from `/kaggle/input`, or enable Internet so the notebook can download `gilkeyio/AudioMNIST`


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "hydra": "hydra-core",
    "omegaconf": "omegaconf",
    "datasets": "datasets",
    "huggingface_hub": "huggingface_hub",
    "sklearn": "scikit-learn",
}

missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("Missing packages installed:" if missing else "All required packages are already installed.", missing)


In [ ]:
from pathlib import Path
import sys


def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]

    kaggle_working = Path("/kaggle/working")
    kaggle_input = Path("/kaggle/input")

    if kaggle_working.exists():
        candidates.append(kaggle_working)
        candidates.extend(kaggle_working.iterdir())

    if kaggle_input.exists():
        candidates.extend(kaggle_input.iterdir())

    for candidate in candidates:
        if not candidate.exists() or not candidate.is_dir():
            continue

        if (candidate / "configs" / "config.yaml").exists() and (candidate / "src").exists():
            return candidate

        nested_candidate = candidate / "speech-emo-recognition"
        if (nested_candidate / "configs" / "config.yaml").exists() and (nested_candidate / "src").exists():
            return nested_candidate

    raise FileNotFoundError(
        "Could not find the project root. Put the repo on Kaggle and make sure it contains configs/config.yaml and src/."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT


In [ ]:
from datasets import load_dataset, load_from_disk

ATTACHED_DATASET_PATH = None
# Example:
# ATTACHED_DATASET_PATH = Path("/kaggle/input/audiomnist-hf/AudioMNIST")

WORKING_DATASET_PATH = Path("/kaggle/working/data/AudioMNIST")

if ATTACHED_DATASET_PATH is not None and Path(ATTACHED_DATASET_PATH).exists():
    dataset_path = Path(ATTACHED_DATASET_PATH)
    load_from_disk(str(dataset_path))
else:
    dataset_path = WORKING_DATASET_PATH
    dataset_path.parent.mkdir(parents=True, exist_ok=True)

    if not dataset_path.exists():
        dataset = load_dataset("gilkeyio/AudioMNIST")
        dataset.save_to_disk(str(dataset_path))
    else:
        load_from_disk(str(dataset_path))

print(f"Dataset path: {dataset_path}")


In [ ]:
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

GlobalHydra.instance().clear()

overrides = [
    f"dataset.paths.local_path={dataset_path.as_posix()}",
    "train.epochs=5",
    "dataloader.batch_size=64",
    "train.log_every=50",
]

with initialize_config_dir(config_dir=str(PROJECT_ROOT / "configs"), version_base="1.3"):
    cfg = compose(config_name="config", overrides=overrides)

print(OmegaConf.to_yaml(cfg))


In [ ]:
import torch.nn as nn

from src.loader import create_dataloaders
from src.model import LSTMClassifier
from src.train import evaluate, train

train_loader, dev_loader, test_loader = create_dataloaders(cfg)
model = LSTMClassifier(cfg)

history = train(cfg, model, train_loader, dev_loader)

device = next(model.parameters()).device
test_metrics = evaluate(
    model,
    test_loader,
    nn.CrossEntropyLoss(),
    device,
    average=cfg.train.metrics_average,
)

print(test_metrics)


In [ ]:
import json
import torch

artifacts_dir = Path("/kaggle/working/artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

torch.save(model.state_dict(), artifacts_dir / "model.pt")

with open(artifacts_dir / "history.json", "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)

with open(artifacts_dir / "test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)

print(f"Artifacts saved to: {artifacts_dir}")
